# Training patch quality and label audit

This notebook is a **read-only audit** of the saved training patches. It is deliberately independent of the training notebook so that inspecting labels does not regenerate or modify them.

It answers four questions:

1. Are image/label files paired, readable, and shaped as expected?
2. Are label values and cross-head relationships internally consistent?
3. Which patches are most suspicious, and why?
4. Do the labels visually agree with the NLS, NPC, and membrane image channels?

The label format is Vulcan 2.5: nine `uint8` heads, with `255` meaning **UNANNOTATED**. For mask heads, `0` means a supervised negative—not missing data. That distinction is shown explicitly in every visual.


## 1. Configuration

Set `PATCH_ROOT` and `LABEL_DIR_NAME` before running the notebook. The defaults point at the current local Vulcan 2.5 ROI pool. To compare historical labels, change only `LABEL_DIR_NAME` (for example, `"labels_fixed"` or `"labels_preDropletFix_bugged"`) and rerun all cells.


In [ ]:
from pathlib import Path
from collections import Counter
import os, re, hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, to_rgb

try:
    from scipy import ndimage
except ImportError:
    ndimage = None

SEED = 42
UNANNOTATED = 255
Z_OFFSET_SCALE = 10.0

PATCH_ROOT = Path(os.environ.get(
    "VULCAN_PATCH_ROOT",
    "/home/tdeibert/Data/Machine_Learning_Dev/Outputs/reviewed_patches_Vulcan_2.5",
))
LABEL_DIR_NAME = os.environ.get("VULCAN_LABEL_DIR", "labels")
IMAGE_DIR = PATCH_ROOT / "images"
LABEL_DIR = PATCH_ROOT / LABEL_DIR_NAME

HEAD_NAMES = [
    "background", "droplet_interior", "droplet_edge", "npc",
    "nucleus_interior", "nucleus_edge", "nucleus_equatorial",
    "abnormal_nucleus", "z_offset",
]
HEAD_INDEX = {name: i for i, name in enumerate(HEAD_NAMES)}
MASK_HEADS = HEAD_NAMES[:-1]

# Stored inputs are five z planes x three channels, ordered per plane.
N_Z_CONTEXT = 5
N_INPUT_CHANNELS = 3
INPUT_NAMES = ["NLS (SV40-NLS)", "NPC (mAb414)", "Membrane (DiO)"]
CENTER_Z = N_Z_CONTEXT // 2
CENTER_SLICE = slice(CENTER_Z * N_INPUT_CHANNELS, (CENTER_Z + 1) * N_INPUT_CHANNELS)

# Every filename is inventoried. Content checks default to a deterministic sample
# because 15k float32 image patches represent hundreds of GB of reads.
# Set to None for an exhaustive content audit.
MAX_FILES = 1500

print("Patch root :", PATCH_ROOT)
print("Images     :", IMAGE_DIR)
print("Labels     :", LABEL_DIR, "<-- verify this is the intended label version")


## 2. Pair inventory

Pairs are matched by the complete filename suffix after `img_` / `lab_`. This catches missing counterparts and avoids the dangerous assumption that two sorted directory listings remain aligned.


In [ ]:
STEM_RE = re.compile(
    r"t(?P<t>\d+)_z(?P<z>\d+)_d(?P<droplet>\d+)_y(?P<cy>\d+)_x(?P<cx>\d+)_"
    r"(?P<tag>[A-Za-z]+)_p(?P<pid>\d+)"
)

def key_for(path):
    name = Path(path).name
    return name[4:-4] if name.endswith(".npy") and name[:4] in {"img_", "lab_"} else None

def parse_key(key):
    m = STEM_RE.fullmatch(key)
    if not m:
        return {"t": np.nan, "z": np.nan, "droplet": np.nan,
                "cy": np.nan, "cx": np.nan, "tag": "unparsed", "pid": np.nan}
    out = m.groupdict()
    for name in ("t", "z", "droplet", "cy", "cx", "pid"):
        out[name] = int(out[name])
    return out

if not IMAGE_DIR.is_dir() or not LABEL_DIR.is_dir():
    raise FileNotFoundError(
        f"Expected image and label directories. Found images={IMAGE_DIR.is_dir()}, "
        f"labels={LABEL_DIR.is_dir()}"
    )

images = {key_for(p): p for p in IMAGE_DIR.glob("img_*.npy")}
labels = {key_for(p): p for p in LABEL_DIR.glob("lab_*.npy")}
image_only = sorted(set(images) - set(labels))
label_only = sorted(set(labels) - set(images))
paired_keys = sorted(set(images) & set(labels))

print(f"images={len(images):,}  labels={len(labels):,}  paired={len(paired_keys):,}")
print(f"images without labels={len(image_only):,}  labels without images={len(label_only):,}")
if image_only[:5]: print("first image-only keys:", image_only[:5])
if label_only[:5]: print("first label-only keys:", label_only[:5])

rng = np.random.default_rng(SEED)
audit_keys = paired_keys
if MAX_FILES is not None and len(audit_keys) > MAX_FILES:
    audit_keys = sorted(rng.choice(audit_keys, size=MAX_FILES, replace=False).tolist())
print(f"auditing {len(audit_keys):,} pairs")


## 3. Structural and semantic audit

The checks below flag strong contradictions, not subjective morphology. Important rules:

- mask heads may contain only `0`, `1`, or `255`;
- image and label height/width must agree;
- background cannot be positive where droplet or nucleus is positive;
- a positive nucleus must be inside a positive droplet;
- equatorial positives must be nucleus positives;
- annotated z-offset pixels must be nucleus positives, and positive nucleus pixels should have z supervision when that head is in use;
- zero-valued nucleus labels inside a droplet are counted separately because they are the most consequential supervised negatives.

Edge and NPC heads are not forced to be subsets of nucleus interior: they describe boundary/shell pixels and can legitimately straddle it.


In [ ]:
def _components(mask):
    if ndimage is None or not mask.any():
        return np.nan
    return int(ndimage.label(mask)[1])

def audit_pair(key):
    ip, lp = images[key], labels[key]
    meta = parse_key(key)
    row = {"key": key, "image_path": str(ip), "label_path": str(lp), **meta}
    flags = []
    try:
        x = np.load(ip, mmap_mode="r")
        y = np.load(lp, mmap_mode="r")
    except Exception as exc:
        row.update(load_error=repr(exc), flags="load_error", flag_count=1)
        return row

    row.update(image_shape=str(tuple(x.shape)), label_shape=str(tuple(y.shape)),
               image_dtype=str(x.dtype), label_dtype=str(y.dtype), load_error="")
    if x.ndim != 3 or x.shape[-1] != N_Z_CONTEXT * N_INPUT_CHANNELS:
        flags.append("unexpected_image_shape")
    if y.ndim != 3 or y.shape[-1] != len(HEAD_NAMES):
        flags.append("unexpected_label_shape")
    if x.ndim < 2 or y.ndim < 2 or x.shape[:2] != y.shape[:2]:
        flags.append("spatial_shape_mismatch")
    # Check the center plane rather than forcing all five mmap'd planes off disk.
    x_check = np.asarray(x[..., CENTER_SLICE]) if x.ndim == 3 and x.shape[-1] >= CENTER_SLICE.stop else x
    if x.dtype.kind not in "fui" or (x.dtype.kind == "f" and not np.isfinite(x_check).all()):
        flags.append("nonfinite_or_nonnumeric_image")
    if y.dtype != np.uint8:
        flags.append("label_not_uint8")
    if y.ndim != 3 or y.shape[-1] < len(HEAD_NAMES):
        row.update(flags=";".join(flags), flag_count=len(flags))
        return row

    h = {name: np.asarray(y[..., i]) for i, name in enumerate(HEAD_NAMES)}
    for name in MASK_HEADS:
        vals = np.unique(h[name])
        invalid = vals[~np.isin(vals, [0, 1, UNANNOTATED])]
        row[f"{name}_pos_frac"] = float((h[name] == 1).mean())
        row[f"{name}_unann_frac"] = float((h[name] == UNANNOTATED).mean())
        if invalid.size:
            flags.append(f"invalid_values:{name}")

    bg = h["background"] == 1
    drop = h["droplet_interior"] == 1
    nuc = h["nucleus_interior"] == 1
    eq = h["nucleus_equatorial"] == 1
    zo_valid = h["z_offset"] != UNANNOTATED
    zo_um = h["z_offset"].astype(np.float32) / Z_OFFSET_SCALE

    row.update(
        bg_drop_overlap_px=int((bg & drop).sum()),
        bg_nucleus_overlap_px=int((bg & nuc).sum()),
        nucleus_outside_droplet_px=int((nuc & ~drop).sum()),
        equatorial_outside_nucleus_px=int((eq & ~nuc).sum()),
        z_annotated_outside_nucleus_px=int((zo_valid & ~nuc).sum()),
        nucleus_without_z_px=int((nuc & ~zo_valid).sum()),
        nucleus_negative_inside_droplet_px=int(((h["nucleus_interior"] == 0) & drop).sum()),
        nucleus_components=_components(nuc),
        z_offset_min_um=float(zo_um[zo_valid].min()) if zo_valid.any() else np.nan,
        z_offset_max_um=float(zo_um[zo_valid].max()) if zo_valid.any() else np.nan,
    )
    for col, threshold, flag in [
        ("bg_drop_overlap_px", 0, "background_overlaps_droplet"),
        ("bg_nucleus_overlap_px", 0, "background_overlaps_nucleus"),
        ("nucleus_outside_droplet_px", 0, "nucleus_outside_droplet"),
        ("equatorial_outside_nucleus_px", 0, "equatorial_outside_nucleus"),
        ("z_annotated_outside_nucleus_px", 0, "z_annotated_outside_nucleus"),
        ("nucleus_without_z_px", 0, "nucleus_without_z_annotation"),
    ]:
        if row[col] > threshold:
            flags.append(flag)
    if not nuc.any():
        flags.append("no_nucleus_positive")
    if nuc.mean() > 0.50:
        flags.append("nucleus_over_half_patch")
    if drop.mean() > 0.98:
        flags.append("droplet_fills_patch")
    row["flags"] = ";".join(flags)
    row["flag_count"] = len(flags)
    return row

rows = []
for i, key in enumerate(audit_keys, 1):
    rows.append(audit_pair(key))
    if i % 1000 == 0:
        print(f"audited {i:,}/{len(audit_keys):,}")

audit = pd.DataFrame(rows)
print(f"done: {len(audit):,} patches; {(audit.flag_count > 0).sum():,} have one or more flags")
audit.head()


## 4. Dataset-level summary and suspicious-patch table

Use the flag census to identify systematic failures. The ranked table is the queue for visual inspection; a flag is a prompt to inspect, not proof that a patch is unusable.


In [ ]:
def flag_census(frame=audit):
    counts = Counter(
        flag for text in frame.loc[frame["flags"].notna(), "flags"]
        for flag in str(text).split(";") if flag
    )
    return pd.DataFrame(counts.most_common(), columns=["flag", "patches"])

display(flag_census())

summary_cols = [
    "nucleus_interior_pos_frac", "nucleus_interior_unann_frac",
    "droplet_interior_pos_frac", "background_pos_frac", "npc_pos_frac",
    "nucleus_negative_inside_droplet_px", "nucleus_components",
    "z_offset_min_um", "z_offset_max_um", "flag_count",
]
display(audit[[c for c in summary_cols if c in audit]].describe(percentiles=[.01,.05,.5,.95,.99]).T)

if "t" in audit:
    display(pd.crosstab(audit["t"], audit["flag_count"] > 0, margins=True)
            .rename(columns={False: "no_flag", True: "flagged"}))

review_queue = audit.sort_values(
    ["flag_count", "nucleus_outside_droplet_px", "nucleus_negative_inside_droplet_px"],
    ascending=False,
).reset_index(drop=True)
display(review_queue[["key", "t", "z", "flag_count", "flags"]].head(50))


## 5. Visual inspection

`show_patch()` displays the center input plane, overlays the two most important masks, then shows every stored head independently. For mask heads: **color = positive, black = supervised negative, gray = UNANNOTATED**. The z panel is gray where unannotated.

Pass either a row number from `review_queue`, a full patch key, or a substring. Start with the default (most strongly flagged patch), then step through the queue.


In [ ]:
HEAD_COLORS = {
    "background": "#e2e8f0", "droplet_interior": "#3b82f6",
    "droplet_edge": "#22d3ee", "npc": "#ffd700",
    "nucleus_interior": "#22c55e", "nucleus_edge": "#bbf7d0",
    "nucleus_equatorial": "#e879f9", "abnormal_nucleus": "#ef4444",
}
ZERO_RGB = np.asarray(to_rgb("#09090b"))
UNANN_RGB = np.asarray(to_rgb("#6b7280"))

def tristate_rgb(channel, color):
    rgb = np.empty(channel.shape + (3,), dtype=np.float32)
    rgb[...] = ZERO_RGB
    rgb[channel == UNANNOTATED] = UNANN_RGB
    rgb[channel == 1] = to_rgb(color)
    return rgb

def resolve_key(which=0):
    if isinstance(which, (int, np.integer)):
        return review_queue.iloc[int(which)].key
    which = str(which)
    if which in images and which in labels:
        return which
    hits = [k for k in paired_keys if which in k]
    if len(hits) != 1:
        raise KeyError(f"Expected one match for {which!r}; found {len(hits)}")
    return hits[0]

def _display_plane(a, lo=1, hi=99.8):
    a = np.asarray(a, dtype=np.float32)
    finite = a[np.isfinite(a)]
    if not finite.size:
        return np.zeros_like(a)
    v0, v1 = np.percentile(finite, [lo, hi])
    return np.clip((a - v0) / max(v1 - v0, 1e-6), 0, 1)

def show_patch(which=0):
    key = resolve_key(which)
    x = np.load(images[key])
    y = np.load(labels[key])
    center = x[..., CENTER_SLICE]
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    ax = axes.ravel()
    for i, name in enumerate(INPUT_NAMES):
        ax[i].imshow(_display_plane(center[..., i]), cmap="gray", vmin=0, vmax=1)
        ax[i].set_title(name)

    nls = _display_plane(center[..., 0])
    ax[3].imshow(nls, cmap="gray", vmin=0, vmax=1)
    drop = np.ma.masked_where(y[..., HEAD_INDEX["droplet_interior"]] != 1,
                             y[..., HEAD_INDEX["droplet_interior"]])
    nuc = np.ma.masked_where(y[..., HEAD_INDEX["nucleus_interior"]] != 1,
                            y[..., HEAD_INDEX["nucleus_interior"]])
    ax[3].imshow(drop, cmap=ListedColormap(["#3b82f6"]), alpha=.25, vmin=0, vmax=1)
    ax[3].imshow(nuc, cmap=ListedColormap(["#22c55e"]), alpha=.55, vmin=0, vmax=1)
    ax[3].set_title("NLS + droplet (blue) + nucleus (green)")

    for j, name in enumerate(HEAD_NAMES):
        a = ax[4 + j]
        ch = y[..., HEAD_INDEX[name]]
        unann = 100 * (ch == UNANNOTATED).mean()
        if name == "z_offset":
            z = ch.astype(np.float32) / Z_OFFSET_SCALE
            z[ch == UNANNOTATED] = np.nan
            cmap = plt.get_cmap("magma").copy(); cmap.set_bad(UNANN_RGB)
            vmax = max(float(np.nanmax(z)) if np.isfinite(z).any() else 1.0, 1.0)
            im = a.imshow(z, cmap=cmap, vmin=0, vmax=vmax)
            fig.colorbar(im, ax=a, fraction=.046, pad=.04, label="um")
            a.set_title(f"z_offset | {unann:.1f}% unann")
        else:
            pos = 100 * (ch == 1).mean()
            a.imshow(tristate_rgb(ch, HEAD_COLORS[name]))
            a.set_title(f"{name}\n{pos:.2f}% pos | {unann:.1f}% unann")
    for a in ax:
        a.axis("off")
    row = audit.loc[audit.key == key].iloc[0]
    fig.suptitle(f"{key}\nflags: {row['flags'] or 'none'}", fontsize=11)
    plt.tight_layout(rect=[0, 0, 1, .96])
    plt.show()
    return row

show_patch(0)


## 6. Contact sheets for pattern finding

These compact views help distinguish isolated defects from systematic labeling errors. Change `query` to a flag name, or set it to `None` for a seeded random sample.


In [ ]:
def contact_sheet(query=None, n=24, seed=SEED):
    frame = audit if query is None else audit[audit["flags"].fillna("").str.contains(query, regex=False)]
    if frame.empty:
        print("No matching patches."); return
    pick = frame.sample(min(n, len(frame)), random_state=seed)
    cols = 6; rows = int(np.ceil(len(pick) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(18, 3 * rows), squeeze=False)
    for a in axes.ravel(): a.axis("off")
    for a, row in zip(axes.ravel(), pick.itertuples()):
        x = np.load(images[row.key], mmap_mode="r")
        y = np.load(labels[row.key], mmap_mode="r")
        a.imshow(_display_plane(x[..., CENTER_SLICE][..., 0]), cmap="gray", vmin=0, vmax=1)
        nuc = y[..., HEAD_INDEX["nucleus_interior"]]
        a.contour(nuc == 1, levels=[.5], colors=["#22c55e"], linewidths=.8)
        drop = y[..., HEAD_INDEX["droplet_interior"]]
        a.contour(drop == 1, levels=[.5], colors=["#3b82f6"], linewidths=.6)
        a.set_title(f"t{int(row.t):03d} z{int(row.z):03d} p{int(row.pid):06d}\n{row.flags[:42]}", fontsize=7)
    fig.suptitle(f"query={query!r} | green=nucleus, blue=droplet", fontsize=12)
    plt.tight_layout(rect=[0, 0, 1, .97]); plt.show()

contact_sheet("nucleus_outside_droplet")


## 7. Image–label alignment cues

Internal consistency cannot prove that a mask follows the biology. This section measures simple, interpretable boundary cues on the center plane:

- nucleus boundary NLS contrast: intensity just inside minus just outside the labeled nuclear boundary;
- NPC-on-rim enrichment: NPC intensity on the labeled nuclear edge relative to the remaining droplet;
- membrane-on-droplet-edge enrichment: membrane intensity on the labeled droplet edge relative to its interior.

These are ranking signals rather than acceptance thresholds. Low or negative values should be inspected visually, especially in early or out-of-focus planes.


In [ ]:
def alignment_cues(key):
    if ndimage is None:
        raise ImportError("scipy is required for boundary alignment cues")
    x = np.load(images[key], mmap_mode="r")[..., CENTER_SLICE]
    y = np.load(labels[key], mmap_mode="r")
    nls, npc_img, mem = [_display_plane(x[..., i]) for i in range(3)]
    nuc = y[..., HEAD_INDEX["nucleus_interior"]] == 1
    drop = y[..., HEAD_INDEX["droplet_interior"]] == 1
    ne = y[..., HEAD_INDEX["nucleus_edge"]] == 1
    de = y[..., HEAD_INDEX["droplet_edge"]] == 1
    er = ndimage.binary_erosion(nuc, iterations=2)
    di = ndimage.binary_dilation(nuc, iterations=2)
    inner, outer = nuc & ~er, di & ~nuc
    safe_mean = lambda a, m: float(a[m].mean()) if m.any() else np.nan
    return {
        "key": key,
        "nls_boundary_contrast": safe_mean(nls, inner) - safe_mean(nls, outer),
        "npc_rim_enrichment": safe_mean(npc_img, ne) - safe_mean(npc_img, drop & ~ne),
        "membrane_edge_enrichment": safe_mean(mem, de) - safe_mean(mem, drop & ~de),
    }

if ndimage is not None:
    cue_keys = audit.loc[audit.load_error.eq(""), "key"].tolist()
    cues = pd.DataFrame(alignment_cues(k) for k in cue_keys)
    audit_with_cues = audit.merge(cues, on="key", how="left")
    display(audit_with_cues[["nls_boundary_contrast", "npc_rim_enrichment",
                             "membrane_edge_enrichment"]].describe(percentiles=[.01,.05,.5,.95,.99]).T)
    display(audit_with_cues.nsmallest(30, "nls_boundary_contrast")[
        ["key", "t", "z", "nls_boundary_contrast", "flags"]])
else:
    print("scipy unavailable; skipping alignment cues")


## 8. Optional report export

Nothing above writes to the patch pool. If you want a durable review queue, uncomment the final line below. The CSV contains paths and diagnostics only; it never edits images or labels.


In [ ]:
REPORT_PATH = PATCH_ROOT / f"patch_label_audit_{LABEL_DIR_NAME}.csv"
print("Would write:", REPORT_PATH)
# audit.to_csv(REPORT_PATH, index=False)


## 9. Prototype: instance-safe droplet correction

This section tests the proposed correction **without changing saved labels**. The saved labels have already unioned the droplet ROIs, so candidate instances are reconstructed from nuclear components plus distance-transform peaks. Each candidate boundary is fit independently with the same three-point RANSAC circle method used by the training pipeline. The fitted instance is then eroded by `EROSION_PX = 10` before interiors are combined, and edge bands are computed per instance before union.

This is deliberately a test harness, not the final generator implementation. The pipeline version should apply the same fit/erode/edge sequence directly to each original droplet ROI before any union operation. Pixels removed only to create artificial separation are treated as unknown by the proposed label patch rather than false background.


In [ ]:
from skimage import measure, morphology, segmentation, feature

EROSION_PX = 10
EDGE_BAND_PX = 3
RANSAC_TOL_PX = 4.0
RANSAC_ITERATIONS = 200
RANSAC_MIN_INLIER_FRAC = 0.50
MIN_SEED_DISTANCE_PX = 30

def _circle_from_3(p1, p2, p3):
    ax, ay = p1; bx, by = p2; cx, cy = p3
    d = 2.0 * (ax*(by-cy) + bx*(cy-ay) + cx*(ay-by))
    if abs(d) < 1e-9:
        return None
    a2, b2, c2 = ax*ax+ay*ay, bx*bx+by*by, cx*cx+cy*cy
    ux = (a2*(by-cy) + b2*(cy-ay) + c2*(ay-by)) / d
    uy = (a2*(cx-bx) + b2*(ax-cx) + c2*(bx-ax)) / d
    return ux, uy, float(np.hypot(ux-ax, uy-ay))

def _fit_circle_kasa(points):
    x, y = points[:, 0], points[:, 1]
    A = np.c_[x, y, np.ones(len(x))]
    b = x*x + y*y
    solution, *_ = np.linalg.lstsq(A, b, rcond=None)
    cx, cy = solution[0] / 2.0, solution[1] / 2.0
    radius = np.sqrt(max(solution[2] + cx*cx + cy*cy, 0.0))
    return cx, cy, radius

def fit_circle_ransac(points, tol_px=RANSAC_TOL_PX, n_iter=RANSAC_ITERATIONS,
                      min_inlier_frac=RANSAC_MIN_INLIER_FRAC, seed=SEED):
    points = np.asarray(points, dtype=float)
    if len(points) < 3:
        return None
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_iter):
        ids = rng.choice(len(points), 3, replace=False)
        circle = _circle_from_3(*points[ids])
        if circle is None:
            continue
        cx, cy, radius = circle
        residual = np.abs(np.hypot(points[:, 0]-cx, points[:, 1]-cy) - radius)
        inliers = residual < tol_px
        if best is None or inliers.sum() > best.sum():
            best = inliers
    required = max(3, int(np.ceil(min_inlier_frac * len(points))))
    if best is None or best.sum() < required:
        return None
    cx, cy, radius = _fit_circle_kasa(points[best])
    residual = np.abs(np.hypot(points[:, 0]-cx, points[:, 1]-cy) - radius)
    inliers = residual < tol_px
    return cx, cy, radius, inliers

def _circle_mask(cx, cy, radius, shape):
    yy, xx = np.ogrid[:shape[0], :shape[1]]
    return (xx-cx)**2 + (yy-cy)**2 <= radius**2

def _edge_band(mask, width=EDGE_BAND_PX):
    if not mask.any():
        return np.zeros_like(mask)
    inner_width = max(1, width // 2)
    outer = morphology.binary_dilation(mask, morphology.disk(inner_width))
    inner = morphology.binary_erosion(mask, morphology.disk(width-inner_width))
    return outer & ~inner


In [ ]:
def reconstruct_droplet_instances(drop_union, nucleus_mask=None, min_seed_distance=MIN_SEED_DISTANCE_PX):
    """Split an already-unioned mask for testing; the final importer will use original ROI instances."""
    drop_union = np.asarray(drop_union, bool)
    if not drop_union.any():
        return np.zeros(drop_union.shape, np.int32)
    distance = ndimage.distance_transform_edt(drop_union)
    peaks = feature.peak_local_max(distance, labels=drop_union, min_distance=min_seed_distance,
                                   exclude_border=False)
    markers = np.zeros(drop_union.shape, np.int32)
    next_id = 1
    used_peaks = set()
    if nucleus_mask is not None and len(peaks):
        nucleus_instances = measure.label(np.asarray(nucleus_mask, bool))
        for region in measure.regionprops(nucleus_instances):
            cy, cx = region.centroid
            order = np.argsort((peaks[:, 0]-cy)**2 + (peaks[:, 1]-cx)**2)
            available = [int(i) for i in order if int(i) not in used_peaks]
            if available:
                i = available[0]; py, px = peaks[i]
                markers[py, px] = next_id; next_id += 1; used_peaks.add(i)
    for i, (cy, cx) in enumerate(peaks):
        if i in used_peaks:
            continue
        occupied = markers > 0
        if occupied.any():
            yy, xx = np.where(occupied)
            if np.min((yy-cy)**2 + (xx-cx)**2) < min_seed_distance**2:
                continue
        markers[cy, cx] = next_id; next_id += 1
    if markers.max() == 0:
        cy, cx = np.unravel_index(np.argmax(distance), distance.shape)
        markers[cy, cx] = 1
    return segmentation.watershed(-distance, markers, mask=drop_union)

def correct_droplet_union(drop_union, nucleus_mask=None, erosion_px=EROSION_PX, seed=SEED):
    """Return separated interior/edge masks and fit diagnostics; never writes to disk."""
    instances = reconstruct_droplet_instances(drop_union, nucleus_mask)
    interiors, edges, fits = [], [], []
    for region in measure.regionprops(instances):
        candidate = instances == region.label
        boundary = segmentation.find_boundaries(candidate, mode="inner")
        yy, xx = np.where(boundary)
        points = np.c_[xx, yy]
        fit = fit_circle_ransac(points, seed=seed + int(region.label))
        prior_radius = np.sqrt(region.area / np.pi)
        accepted = False
        reason = "ransac_failed"
        if fit is not None:
            cx, cy, radius, inliers = fit
            center_shift = np.hypot(cx-region.centroid[1], cy-region.centroid[0])
            accepted = (0.5*prior_radius <= radius <= 2.0*prior_radius and
                        center_shift <= prior_radius and inliers.mean() >= RANSAC_MIN_INLIER_FRAC)
            reason = "accepted" if accepted else "geometry_gate"
        if accepted:
            fitted = _circle_mask(cx, cy, radius, candidate.shape)
        else:
            fitted = ndimage.binary_fill_holes(candidate)
            cy, cx = region.centroid; radius = prior_radius
            inliers = np.zeros(len(points), bool)
        interior = morphology.binary_erosion(fitted, morphology.disk(erosion_px))
        if not interior.any():
            reason = "erosion_empty"; accepted = False
            continue
        interiors.append(interior)
        edges.append(_edge_band(interior))
        fits.append(dict(instance=int(region.label), accepted=accepted, reason=reason,
                         cx=float(cx), cy=float(cy), radius=float(radius),
                         prior_radius=float(prior_radius), inlier_fraction=float(inliers.mean())))
    interior_union = np.logical_or.reduce(interiors) if interiors else np.zeros_like(drop_union, bool)
    edge_union = np.logical_or.reduce(edges) if edges else np.zeros_like(drop_union, bool)
    return dict(instances=instances, interior=interior_union, edge=edge_union, fits=pd.DataFrame(fits))

def proposed_droplet_channels(label_patch, correction):
    """Preview tri-state channels. Separation pixels are unknown, never false background."""
    old_drop = label_patch[..., HEAD_INDEX["droplet_interior"]]
    old_bg = label_patch[..., HEAD_INDEX["background"]]
    known = (old_drop != UNANNOTATED) | (old_bg != UNANNOTATED)
    d = np.full(old_drop.shape, UNANNOTATED, np.uint8)
    b = np.full(old_bg.shape, UNANNOTATED, np.uint8)
    e = np.full(old_drop.shape, UNANNOTATED, np.uint8)
    explicit_bg = old_bg == 1
    d[explicit_bg] = 0; b[explicit_bg] = 1; e[explicit_bg] = 0
    corrected_interior = correction["interior"] & ~explicit_bg
    corrected_edge = correction["edge"] & ~explicit_bg
    d[corrected_interior] = 1; b[corrected_interior] = 0
    e[corrected_interior & known] = 0
    e[corrected_edge] = 1
    return {"background": b, "droplet_interior": d, "droplet_edge": e}


### Synthetic acceptance tests

The synthetic case deliberately contains two overlapping circles and a large internal void. The input union has one connected component. Passing requires two separated corrected interiors, two accepted RANSAC fits, retained per-instance edge supervision, and no conversion of the artificial gap into background.


In [ ]:
shape = (256, 256)
left = _circle_mask(76, 128, 55, shape)
right = _circle_mask(180, 128, 55, shape)
joined_with_void = (left | right) & ~_circle_mask(74, 112, 14, shape)
seeds = _circle_mask(76, 128, 8, shape) | _circle_mask(180, 128, 8, shape)
synthetic = correct_droplet_union(joined_with_void, seeds)

assert measure.label(joined_with_void).max() == 1, "fixture must begin joined"
assert measure.label(synthetic["interior"]).max() == 2, "corrected interiors must separate"
assert len(synthetic["fits"]) == 2 and synthetic["fits"]["accepted"].all(), "both fits must pass"
assert synthetic["edge"].any(), "instance edge supervision must survive"
assert np.allclose(sorted(synthetic["fits"]["radius"]), [55, 55], atol=5), "radius recovery failed"
print("Synthetic droplet correction tests: PASS")
display(synthetic["fits"])

fig, ax = plt.subplots(1, 4, figsize=(15, 4))
ax[0].imshow(joined_with_void); ax[0].set_title("input: joined + void")
ax[1].imshow(synthetic["instances"]); ax[1].set_title("reconstructed instances")
ax[2].imshow(synthetic["interior"]); ax[2].set_title("RANSAC + 10 px erosion")
ax[3].imshow(synthetic["edge"]); ax[3].set_title("per-instance edges")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()


### Test on saved training patches

Reconstruction from a saved union mask is retained only as an exploratory visualization. **Multiple nuclei do not imply multiple droplets**, so it must not be used to generate corrected labels. The acceptance test in the next section operates on the original droplet ROIs, where instance identity is still intact.


In [ ]:
def find_crowded_patches(max_results=50):
    rows = []
    for key in audit["key"]:
        y = np.load(labels[key], mmap_mode="r")
        drop = y[..., HEAD_INDEX["droplet_interior"]] == 1
        nuc = y[..., HEAD_INDEX["nucleus_interior"]] == 1
        n_drop = int(measure.label(drop).max())
        n_nuc = int(measure.label(nuc).max())
        if n_drop == 1 and n_nuc >= 2:
            corr = correct_droplet_union(drop, nuc)
            rows.append(dict(key=key, nucleus_components=n_nuc, old_droplet_components=n_drop,
                             reconstructed_instances=int(corr["instances"].max()),
                             corrected_components=int(measure.label(corr["interior"]).max()),
                             accepted_fits=int(corr["fits"]["accepted"].sum()) if len(corr["fits"]) else 0))
    return pd.DataFrame(rows).sort_values(["nucleus_components", "accepted_fits"], ascending=False).head(max_results) if rows else pd.DataFrame()

# Deliberately not run automatically: multiple nuclei can belong to one droplet.
crowded = pd.DataFrame()

def compare_droplet_correction(which=0):
    if crowded.empty:
        print("No sampled patch has multiple nuclei inside one connected droplet label."); return None
    key = crowded.iloc[int(which)]["key"] if isinstance(which, (int, np.integer)) else resolve_key(which)
    x = np.load(images[key], mmap_mode="r")[..., CENTER_SLICE]
    y = np.load(labels[key])
    drop = y[..., HEAD_INDEX["droplet_interior"]] == 1
    nuc = y[..., HEAD_INDEX["nucleus_interior"]] == 1
    corr = correct_droplet_union(drop, nuc)
    proposed = proposed_droplet_channels(y, corr)
    fig, ax = plt.subplots(2, 3, figsize=(15, 10))
    ax[0,0].imshow(_display_plane(x[..., 2]), cmap="gray"); ax[0,0].contour(drop, [.5], colors="red"); ax[0,0].set_title("membrane + current droplet")
    ax[0,1].imshow(drop); ax[0,1].set_title(f"current interior: {measure.label(drop).max()} component(s)")
    ax[0,2].imshow(corr["instances"]); ax[0,2].set_title("candidate instances")
    ax[1,0].imshow(corr["interior"]); ax[1,0].set_title(f"corrected interior: {measure.label(corr['interior']).max()} component(s)")
    ax[1,1].imshow(corr["edge"]); ax[1,1].set_title("per-instance corrected edges")
    ax[1,2].imshow(tristate_rgb(proposed["droplet_interior"], HEAD_COLORS["droplet_interior"])); ax[1,2].set_title("proposed tri-state interior")
    for a in ax.ravel(): a.axis("off")
    fig.suptitle(key); plt.tight_layout(rect=[0,0,1,.96]); plt.show()
    display(corr["fits"]); return corr, proposed

# To inspect this diagnostic manually: crowded = find_crowded_patches(); display(crowded)


### Instance-preserving acceptance test on the original droplet ROIs

This is the decisive test. It reads `DropletRoiSet.zip`, retains every ROI as a separate instance, fits each polygon with RANSAC, erodes each fitted radius by 10 pixels, and only then unions the masks. Connectivity is measured at quarter resolution to keep memory bounded; all circle fits use the original full-resolution coordinates. No labels are written.


In [ ]:
from collections import defaultdict
from skimage import draw as skdraw
import roifile

DROPLET_ROI_ZIP = Path("/data/user/tdeibert/Nuclear_Scaling/Inputs/ROIS/DropletRoiSet.zip")

def load_droplet_roi_instances(path=DROPLET_ROI_ZIP):
    frames = defaultdict(list)
    for roi in roifile.roiread(str(path)):
        polygon = np.asarray(roi.coordinates(), dtype=float)
        if polygon.ndim != 2 or polygon.shape[0] < 3 or polygon.shape[1] != 2:
            continue
        frame = (int((roi.t_position or 1)-1), int((roi.z_position or 1)-1))
        frames[frame].append(dict(name=str(roi.name), polygon=polygon))
    return frames

def test_original_droplet_rois(path=DROPLET_ROI_ZIP, scale=4.0, n_iter=80):
    frames = load_droplet_roi_instances(path)
    rows, details = [], {}
    for (t, z), items in sorted(frames.items()):
        max_x = int(max(it["polygon"][:,0].max() for it in items) / scale) + 3
        max_y = int(max(it["polygon"][:,1].max() for it in items) / scale) + 3
        original = np.zeros((max_y, max_x), bool)
        corrected = np.zeros_like(original)
        fits = []
        for index, item in enumerate(items):
            polygon = item["polygon"]
            q = polygon / scale
            rr, cc = skdraw.polygon(q[:,1], q[:,0], shape=original.shape)
            original[rr, cc] = True
            fit = fit_circle_ransac(polygon, n_iter=n_iter, seed=SEED+index)
            if fit is None:
                fits.append(dict(name=item["name"], accepted=False, reason="ransac_failed"))
                continue
            cx, cy, radius, inliers = fit
            accepted = bool(inliers.mean() >= RANSAC_MIN_INLIER_FRAC and radius > EROSION_PX)
            fits.append(dict(name=item["name"], accepted=accepted, reason="accepted" if accepted else "geometry_gate",
                             cx=float(cx), cy=float(cy), radius=float(radius),
                             inlier_fraction=float(inliers.mean())))
            if not accepted:
                continue
            cx_s, cy_s = cx/scale, cy/scale
            eroded_radius = (radius-EROSION_PX)/scale
            y0=max(0,int(cy_s-eroded_radius)); y1=min(max_y,int(cy_s+eroded_radius)+2)
            x0=max(0,int(cx_s-eroded_radius)); x1=min(max_x,int(cx_s+eroded_radius)+2)
            yy, xx = np.ogrid[y0:y1, x0:x1]
            corrected[y0:y1,x0:x1] |= (xx-cx_s)**2 + (yy-cy_s)**2 <= eroded_radius**2
        original_components = int(measure.label(original, connectivity=2).max())
        corrected_components = int(measure.label(corrected, connectivity=2).max())
        accepted_count = sum(f["accepted"] for f in fits)
        rows.append(dict(t=t, z=z, rois=len(items), accepted_fits=accepted_count,
                         original_components=original_components, corrected_components=corrected_components,
                         original_joined_excess=max(0,len(items)-original_components),
                         corrected_joined_excess=max(0,accepted_count-corrected_components)))
        details[(t,z)] = dict(items=items, fits=pd.DataFrame(fits), original=original, corrected=corrected, scale=scale)
    result = pd.DataFrame(rows)
    return result, details

roi_test, roi_test_details = test_original_droplet_rois()
roi_summary = {
    "frames": len(roi_test), "rois": int(roi_test.rois.sum()),
    "accepted_fits": int(roi_test.accepted_fits.sum()),
    "original_joined_excess": int(roi_test.original_joined_excess.sum()),
    "corrected_joined_excess": int(roi_test.corrected_joined_excess.sum()),
    "frames_improved": int((roi_test.corrected_components > roi_test.original_components).sum()),
    "frames_worse": int((roi_test.corrected_components < roi_test.original_components).sum()),
}
display(pd.Series(roi_summary, name="value").to_frame())
display(roi_test.sort_values("original_joined_excess", ascending=False).head(15))

assert roi_summary["accepted_fits"] / roi_summary["rois"] >= 0.95, "RANSAC acceptance below 95%"
assert roi_summary["corrected_joined_excess"] <= max(1, int(.01*roi_summary["accepted_fits"])), "too many fitted droplets remain joined"
assert roi_summary["frames_worse"] == 0, "correction reduced component count in at least one frame"
assert roi_summary["frames_improved"] > 0, "correction did not separate any joined frames"
print("Original-ROI droplet correction tests: PASS")


In [ ]:
def show_roi_frame_correction(t=None, z=None):
    if t is None or z is None:
        row = roi_test.assign(gain=roi_test.corrected_components-roi_test.original_components).nlargest(1, "gain").iloc[0]
        t, z = int(row.t), int(row.z)
    detail = roi_test_details[(t,z)]
    fig, ax = plt.subplots(1, 2, figsize=(14, 7))
    ax[0].imshow(detail["original"], cmap="gray"); ax[0].set_title(f"Original ROI union: {measure.label(detail['original']).max()} components")
    ax[1].imshow(detail["corrected"], cmap="gray"); ax[1].set_title(f"RANSAC + 10 px per instance: {measure.label(detail['corrected']).max()} components")
    for a in ax: a.axis("off")
    fig.suptitle(f"Droplet ROI connectivity, t={t}, z={z} (display at 1/{detail['scale']:g} resolution)")
    plt.tight_layout(); plt.show()
    return roi_test.loc[(roi_test.t==t)&(roi_test.z==z)], detail["fits"]

roi_frame_summary, roi_frame_fits = show_roi_frame_correction()
display(roi_frame_summary)
